In [ ]:
#Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#Download the loans dataset:
!gdown '1y4WkBtd7p_fPUCz-SaZ9t4fdg3TFSGZs'

#Source: https://github.com/gedeck/practical-statistics-for-data-scientists/

In [ ]:
#Load data into a DataFrame
df=pd.read_csv('loan3000.csv')

In [ ]:
#Inspect the dataset:
df.head(10)

In [ ]:
df.describe().T

In [ ]:
#The scales of the continuous variables differ significantly. Therefore, we normalize these columns.
#Normalization: set min. = 0, max =1

normalize_columns=['borrower_score','dti','payment_inc_ratio']
for col in normalize_columns:
  df[col]=(df[col]-df[col].min())/(df[col].max()-df[col].min())


In [ ]:
def default(x):
  if x=='default':
    return 1
  else:
    return 0

In [ ]:
df['default']=df['outcome'].apply(default)

In [ ]:
df.head(10)

In [ ]:
#Create dummy variables for the purpose column:
df=pd.get_dummies(df, columns=['purpose_'])

In [ ]:
df.head()

In [ ]:
df=df.drop(columns=['Unnamed: 0','purpose__other','outcome'])
#Unnamed: 0 = loan ids
#New dummy variable was created for outcome, hence 'outcome' can be removed
#Drop one of the purpose dummies to prevent multicollinearity

In [ ]:
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

#First separate the outcome variable from the features
y=df['default']
df=df.drop(columns='default')

#Next, split the dataset into a training set and a test set
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.15)

In [ ]:
#k Nearest Neighbors with k=5:

from sklearn.neighbors import KNeighborsClassifier
knn=KNeighborsClassifier(n_neighbors=5)

#Model accuracy:
knn.fit(X_train,y_train)
knn.score(X_test,y_test)

In [ ]:
#Logistic regression
from sklearn.linear_model import LogisticRegression

logistic = LogisticRegression()
logistic.fit(X_train,y_train)
logistic.score(X_train,y_train)

In [ ]:
logistic.score(X_test,y_test)

In [ ]:
#Decision tree:
from sklearn.tree import DecisionTreeClassifier
dtc = DecisionTreeClassifier()
dtc.fit(X_train,y_train)
dtc.score(X_train,y_train)

In [ ]:
dtc.score(X_test,y_test)

In [ ]:
from sklearn import tree
fig = plt.figure(figsize=(25,20))
_ = tree.plot_tree(dtc,
    feature_names=X_train.columns.to_list(),
    class_names=['paid off','default'],
    filled=True)


In [ ]:
dtc2 = DecisionTreeClassifier(max_depth=4)
dtc2.fit(X_train,y_train)
dtc2.score(X_train,y_train)


In [ ]:
dtc2.score(X_test,y_test)

In [ ]:
#Display the decision tree
from sklearn import tree
fig = plt.figure(figsize=(10,5))
_ = tree.plot_tree(dtc2,
                   feature_names=X_train.columns.to_list(),
                   class_names=['paid off','default'],
                   filled=True)

In [ ]:
#Neural network with default settings:
from sklearn.neural_network import MLPClassifier
mlp=MLPClassifier()
mlp.fit(X_train,y_train)

#Model accuracy on training data:
mlp.score(X_train,y_train)

In [ ]:
#Model accuracy on test data:
mlp.score(X_test,y_test)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

#knn
y_pred=knn.predict(X_test)
print('KNN:')
print(f'Precision: {precision_score(y_test, y_pred):.3f}' )
print(f'Recall   : {recall_score(y_test, y_pred):.3f}\n' )

#logistic regression
y_pred=logistic.predict(X_test)
print('Logistic regression:')
print(f'Precision: {precision_score(y_test, y_pred):.3f}' )
print(f'Recall   : {recall_score(y_test, y_pred):.3f}\n' )

#Decision tree
y_pred=dtc.predict(X_test)
print('Decision tree:')
print(f'Precision: {precision_score(y_test, y_pred):.3f}' )
print(f'Recall   : {recall_score(y_test, y_pred):.3f}\n' )

#Decision tree (max depth)
y_pred=dtc2.predict(X_test)
print('Decision tree (max depth=4):')
print(f'Precision: {precision_score(y_test, y_pred):.3f}' )
print(f'Recall   : {recall_score(y_test, y_pred):.3f}\n' )

#Neural network
y_pred=mlp.predict(X_test)
print('Neural network:')
print(f'Precision: {precision_score(y_test, y_pred):.3f}' )
print(f'Recall   : {recall_score(y_test, y_pred):.3f}\n' )

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots()

RocCurveDisplay.from_estimator(knn, X_test, y_test, name="KNN", ax=ax)
RocCurveDisplay.from_estimator(logistic, X_test, y_test, name="Logistic", ax=ax)
RocCurveDisplay.from_estimator(dtc, X_test, y_test, name="Decision Tree", ax=ax)
RocCurveDisplay.from_estimator(dtc2, X_test, y_test, name="DT (depth=4)", ax=ax)
RocCurveDisplay.from_estimator(mlp, X_test, y_test, name="Neural Network", ax=ax)

# Diagonal reference line
ax.plot([0, 1], [0, 1], linestyle='--')

ax.set_title("ROC Curves")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate (Recall)")

plt.show()